# 06. Joins, Unions & Data Combining

Combining datasets is the core of relational data processing. In a distributed engine like PySpark, joins can be expensive if not done correctly.

### Topics Covered:
1. **Relational Join Types Explained (Inner, Left, Right, Full Outer, Semi, Anti, Cross)**
2. **Handling Duplicate Column Names (The common ambiguity bug)**
3. **Combining Data with `union()` vs `unionByName()`**
4. **Broadcast Joins: Eliminating network shuffles for 10x-100x speedup!**
5. **Stopping the SparkSession to free memory**


---
## 1. Initializing SparkSession


In [ ]:


from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast

spark = SparkSession.builder \
    .appName("06_Joins_and_Combining") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession ready!")


---
## 2. Creating Sample Datasets: Employees & Departments


In [ ]:
# Employees table (emp_id, emp_name, dept_id, salary)
employees_data = [
    (1, "Alice", 10, 85000),
    (2, "Bob", 20, 62000),
    (3, "Charlie", 10, 92000),
    (4, "Diana", 30, 58000),
    (5, "Evan", 99, 70000)  # dept_id 99 does NOT exist in departments table
]
emp_cols = ["emp_id", "emp_name", "dept_id", "salary"]
emp_df = spark.createDataFrame(employees_data, emp_cols)

# Departments table (dept_id, dept_name, location)
departments_data = [
    (10, "Engineering", "San Francisco"),
    (20, "Marketing", "New York"),
    (30, "HR", "Chicago"),
    (40, "Finance", "London")  # Finance has NO employees
]
dept_cols = ["dept_id", "dept_name", "location"]
dept_df = spark.createDataFrame(departments_data, dept_cols)

print("Employees DataFrame:")
emp_df.show()

print("Departments DataFrame:")
dept_df.show()


---
## 3. Join Types Visualized & Executed

| Join Type | Result |
| :--- | :--- |
| **`inner`** | Only rows where `dept_id` matches in BOTH tables |
| **`left`** | ALL rows from left (employees); nulls for non-matching right |
| **`right`** | ALL rows from right (departments); nulls for non-matching left |
| **`outer` / `full`** | ALL rows from both tables; nulls wherever there is no match |
| **`left_semi`** | Keeps left rows that HAVE a match in right (does NOT add right columns) |
| **`left_anti`** | Keeps left rows that DO NOT have a match in right (find orphans) |
| **`cross`** | Cartesian product (every left row paired with every right row) |


In [ ]:
# 1. INNER JOIN: Only employees with valid departments (Evan is excluded; Finance is excluded)
print("--- 1. INNER JOIN ---")
emp_df.join(dept_df, "dept_id", "inner").show()


In [ ]:
# 2. LEFT JOIN: All employees (Evan is included with null department info)
print("--- 2. LEFT JOIN ---")
emp_df.join(dept_df, "dept_id", "left").show()


In [ ]:
# 3. RIGHT JOIN: All departments (Finance is included with null employee info)
print("--- 3. RIGHT JOIN ---")
emp_df.join(dept_df, "dept_id", "right").show()


In [ ]:
# 4. FULL OUTER JOIN: All employees AND all departments
print("--- 4. FULL OUTER JOIN ---")
emp_df.join(dept_df, "dept_id", "outer").show()


In [ ]:
# 5. LEFT SEMI JOIN: Returns employees who belong to an existing department
# Notice: No columns from dept_df are added! It acts like a fast SQL "WHERE EXISTS"
print("--- 5. LEFT SEMI JOIN ---")
emp_df.join(dept_df, "dept_id", "left_semi").show()


In [ ]:
# 6. LEFT ANTI JOIN: Returns employees with NO matching department (Find orphan/invalid records!)
print("--- 6. LEFT ANTI JOIN ---")
emp_df.join(dept_df, "dept_id", "left_anti").show()


---
## 4. Handling Column Name Ambiguity

If you join using an expression like `emp_df.dept_id == dept_df.dept_id`, Spark keeps BOTH `dept_id` columns in the output DataFrame.
Subsequent calls to `df.select("dept_id")` will **fail with an AnalysisException (ambiguous column)**!

### Best Practices:
1. Join on column name as string: `emp_df.join(dept_df, "dept_id")` (Spark deduplicates the join column).
2. Or alias DataFrames before joining.


In [ ]:
# Aliasing tables to avoid ambiguity
e = emp_df.alias("e")
d = dept_df.alias("d")

joined = e.join(d, col("e.dept_id") == col("d.dept_id"), "inner")
# Now you can explicitly specify e.dept_id
joined.select("e.emp_name", "e.dept_id", "d.dept_name").show()


---
## 5. Combining DataFrames: `union()` vs `unionByName()`

* `union()` combines rows based on **column position** (Column 1 matches Column 1, regardless of name). If column order differs, data becomes silently corrupted!
* `unionByName()` combines rows based on **column name** (Safe and recommended). It also supports `allowMissingColumns=True`!


In [ ]:
# Batch 1 of new employees
batch1 = spark.createDataFrame([(6, "Grace", 20), (7, "Henry", 10)], ["emp_id", "emp_name", "dept_id"])

# Batch 2 has columns in DIFFERENT order and an extra column
batch2 = spark.createDataFrame([("Isla", 30, 8, "Contractor")], ["emp_name", "dept_id", "emp_id", "role"])

# unionByName safely matches columns by name and fills missing with null
combined_df = batch1.unionByName(batch2, allowMissingColumns=True)
print("Combined DataFrame with unionByName:")
combined_df.show()


---
## 6. Optimization: Broadcast Joins

### What is a Shuffle Hash Join?
When joining two large tables, Spark hashes the join key and sends rows with the same key across the network to the same executor. This **Network Shuffle** is the most expensive operation in big data!

### What is a Broadcast Join?
If one table is small (e.g. lookup table, zip code directory, store catalog < 10MB - 100MB):
* Spark can copy the entire small table to **every executor's memory**.
* The large table stays where it is! Each executor joins locally without any network shuffle.
* Result: **Massive speedup!**


In [ ]:
# Explicitly tell Spark to broadcast the departments lookup table
fast_joined = emp_df.join(broadcast(dept_df), "dept_id", "inner")

fast_joined.show()

# We can inspect the physical execution plan: notice "BroadcastHashJoin" instead of "SortMergeJoin"
print("Physical Execution Plan:")
fast_joined.explain()


---
## 7. Summary of Notebook 06
* Choose the appropriate join: `inner`, `left`, `right`, `outer`, `left_semi`, `left_anti`.
* Avoid duplicate column bugs by joining on column strings (`"dept_id"`) or using aliases (`alias("e")`).
* Use `unionByName(..., allowMissingColumns=True)` to safely combine DataFrames.
* Use `broadcast(small_df)` to eliminate shuffles and dramatically speed up joins.

---
## ⚠️ CRITICAL STEP: Stopping the SparkSession
Always release RAM and JVM background processes before moving to the next notebook.


In [ ]:
# Stop SparkSession to free memory
spark.stop()
print("SparkSession stopped successfully! RAM released.")
